---
title: Log returns
execute:
  enabled: true
---

`q.indicator.log_returns` calculates consecutive $\log(P_t)-\log(P_{t-1})$ values from a one-dimensional positive price array. It returns a NumPy array with one fewer observation than the input.

The example reconstructs a dated `Series` for AAPL and SPY, then compares their rolling 21-session log returns over the latest five shared years.

In [1]:
import pandas as pd

import qrt as q

prices = pd.concat(
    {
        "AAPL": q.data.datasets.load("aapl")["close"],
        "SPY": q.data.datasets.load("spy")["close"],
    },
    axis=1,
    join="inner",
).dropna()
end = prices.index.max()
prices = prices.loc[end - pd.DateOffset(years=5) :]
prices.tail()

,AAPL,SPY
datetime,,
2026-07-14,314.859985,751.830017
2026-07-15,327.500000,754.809998
2026-07-16,333.260010,750.719971
2026-07-17,333.739990,743.289978
2026-07-20,326.589996,742.090027


## Calculate the indicator

Attach the output to `prices.index[1:]` because the first price has no preceding observation.

In [2]:
returns = pd.DataFrame(
    {
        symbol: pd.Series(
            q.indicator.log_returns(prices[symbol]),
            index=prices.index[1:],
        )
        for symbol in prices
    }
)
returns.tail()

,AAPL,SPY
datetime,,
2026-07-14,-0.007751,0.003544
2026-07-15,0.039360,0.003956
2026-07-16,0.017435,-0.005433
2026-07-17,0.001439,-0.009946
2026-07-20,-0.021657,-0.001616


## Compare with SPY

Summing daily log returns produces the cumulative log return over the selected 21-session window. Multiplying by 100 expresses it in log-percentage points.

In [3]:
window = 21
comparison = returns.rolling(window).sum().mul(100)
comparison.columns = [f"{symbol} {window}-session log return" for symbol in comparison]
fig = q.plot.col(
    comparison,
    title=f"AAPL and SPY {window}-session log returns",
    ylabel="Log return (%)",
)
fig.show(renderer="notebook_connected")